# Export TinyLlama HelpSteer2 Training Curves

This notebook exports reproducible training-loss, evaluation-loss, and learning-rate plots from the TinyLlama HelpSteer2 CSV training logs. It runs without a GPU, model weights, adapters, ArmoRM, or a TensorBoard server.

## 1. Clone or update the repository

In [ ]:
%cd /content
import os
import shutil

repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"

if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

## 2. Install lightweight dependencies

The pandas version matches the standard Google Colab environment.

In [ ]:
%pip install -q "pandas==2.2.2" matplotlib

## 3. Show the repository and training logs

In [ ]:
!pwd
!ls
!ls scripts
!ls results || true
!ls results/tinyllama_helpsteer2_training_logs || true

## 4. Required input and expected output

Required input:

- `results/tinyllama_helpsteer2_training_logs/*.csv`

Expected output:

- `results/plots/tensorboard/*.png`
- matching CSV exports in `results/plots/tensorboard/`

## 5. Optional: upload CSV logs manually

Run this cell when the CSV logs are stored on your computer. Select one or more training-log CSV files; they will be placed in the required input folder.

In [ ]:
from pathlib import Path

log_dir = Path("results/tinyllama_helpsteer2_training_logs")
log_dir.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files

    uploaded = files.upload()
    for filename, content in uploaded.items():
        destination = log_dir / Path(filename).name
        destination.write_bytes(content)
        print(f"Saved {destination}")
except ImportError:
    print("Manual upload is available when this notebook runs in Google Colab.")

## 6. Optional: copy CSV logs from Google Drive

Run this section only when your logs are stored in Google Drive. Change `drive_log_dir` to the folder containing your CSV files.

In [ ]:
from pathlib import Path
import shutil

try:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_log_dir = Path("/content/drive/MyDrive/tinyllama_helpsteer2_training_logs")
    destination_dir = Path("results/tinyllama_helpsteer2_training_logs")
    destination_dir.mkdir(parents=True, exist_ok=True)

    csv_files = sorted(drive_log_dir.glob("*.csv")) if drive_log_dir.exists() else []
    if not csv_files:
        print(f"No CSV files found in {drive_log_dir}. Update drive_log_dir if needed.")
    for source in csv_files:
        destination = destination_dir / source.name
        shutil.copy2(source, destination)
        print(f"Copied {source.name} to {destination}")
except ImportError:
    print("Google Drive mounting is available when this notebook runs in Google Colab.")

## 7. Confirm the available CSV logs

In [ ]:
!find results/tinyllama_helpsteer2_training_logs -maxdepth 1 -type f -name "*.csv" -print

## 8. Configure and export the training curves

Individual plots use a TensorBoard-style presentation: a darker raw curve with a white EMA-smoothed curve on top. Change the settings below before exporting. `SMOOTHING=0` disables smoothing, while values closer to `1` produce stronger smoothing. Combined plots keep distinct colors for the five attributes.

In [ ]:
SMOOTHING = 0.5
THEME = "dark"
RAW_COLOR = "#65717f"
SMOOTHED_COLOR = "#ffffff"
RAW_ALPHA = 0.55
RAW_LINEWIDTH = 1.0
SMOOTHED_LINEWIDTH = 2.0
GRID_ALPHA = 0.3
DPI = 200

!python scripts/export_tinyllama_training_curves.py \
  --log_dir results/tinyllama_helpsteer2_training_logs \
  --output_dir results/plots/tensorboard \
  --smoothing {SMOOTHING} \
  --theme {THEME} \
  --raw_color "{RAW_COLOR}" \
  --smoothed_color "{SMOOTHED_COLOR}" \
  --raw_alpha {RAW_ALPHA} \
  --raw_linewidth {RAW_LINEWIDTH} \
  --smoothed_linewidth {SMOOTHED_LINEWIDTH} \
  --grid_alpha {GRID_ALPHA} \
  --dpi {DPI}

## 9. List the generated plots

In [ ]:
!find results/plots/tensorboard -maxdepth 1 -type f -name "*.png" -print

## 10. Display the generated plots

In [ ]:
from pathlib import Path
from IPython.display import Image, display

plot_paths = sorted(Path("results/plots/tensorboard").glob("*.png"))
if not plot_paths:
    print("No PNG plots were found. Check the export output above.")
for plot_path in plot_paths:
    print(plot_path.name)
    display(Image(filename=str(plot_path)))

## 11. Create a zip archive for download

This archive contains the generated plots and their matching CSV exports. Keep the zip file out of GitHub.

In [ ]:
!zip -r tinyllama_training_curves_png.zip results/plots/tensorboard/

## 12. Download the zip archive

In [ ]:
try:
    from google.colab import files

    files.download("tinyllama_training_curves_png.zip")
except ImportError:
    print("Automatic download is available when this notebook runs in Google Colab.")

## 13. Git safety check

Generated PNG files should only be committed when intentionally selected for the thesis or a report. Keep zip files, adapters, checkpoints, `.safetensors`, `.bin`, and model weights out of GitHub.

In [ ]:
!git status